# Motor Exercise 7 — plotting and comparing step responses

Calculate wheel speed from a timestamped step test, inspect individual trials, and compare simple instantaneous, slew-rate and first-order-lag candidate responses.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the plotting route; it is not evidence about your robot and is not a result you should expect to reproduce. When you are ready, change only the settings in **Use the example or your own data** and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV, set it to `False`, and enter the filename. This is the main cell you need to edit.

Expected CSV columns: `trial_id`, `split`, `wheel`, `direction`, `sample_num`, `timestamp_ms`, `step_timestamp_ms`, `PWM`, `encoder_count`, and `phase`.

Use `initial_hold`, `transient` and `final_hold` as phase labels. A PWM step is a request; encoder counts provide the recorded wheel response.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "motor_exercise07_step_responses.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The example contains two repeated rising steps and one held-back falling step. It is designed to exercise the plotting route, not to identify your robot's dynamics.


In [ ]:
example_rows = []
trial_settings = [
    ("rise_1", "fit", 150.0, 790.0, 35, 105),
    ("rise_2", "fit", 150.0, 790.0, 35, 105),
    ("fall_test", "held back", 700.0, 260.0, 95, 45),
]

for trial_index, (trial_id, split, initial, final, pwm_0, pwm_1) in enumerate(trial_settings):
    intervals_ms = rng.integers(38, 44, 58)
    timestamp_ms = np.cumsum(intervals_ms)
    step_index = 6
    step_timestamp_ms = timestamp_ms[step_index]
    elapsed_after_step_s = (timestamp_ms - step_timestamp_ms) / 1000
    response = np.where(
        elapsed_after_step_s < 0,
        initial,
        initial + (final - initial) * (
            1 - np.exp(-np.maximum(elapsed_after_step_s, 0) / 0.18)
        ),
    )
    response += 5 * np.sin(np.arange(len(response)) * 0.8 + trial_index)
    encoder_count = np.rint(
        np.cumsum(response * intervals_ms / 1000)
    ).astype(int)

    for sample_num in range(len(timestamp_ms)):
        if sample_num < step_index:
            phase = "initial_hold"
        elif elapsed_after_step_s[sample_num] > 1.4:
            phase = "final_hold"
        else:
            phase = "transient"
        example_rows.append({
            "trial_id": trial_id,
            "split": split,
            "wheel": "left",
            "direction": "forward",
            "sample_num": sample_num,
            "timestamp_ms": int(timestamp_ms[sample_num]),
            "step_timestamp_ms": int(step_timestamp_ms),
            "PWM": pwm_0 if sample_num < step_index else pwm_1,
            "encoder_count": int(encoder_count[sample_num]),
            "phase": phase,
        })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

This is where your uploaded CSV enters the notebook. Check the first rows before continuing: column names, units and labels should match the exercise.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Calculate elapsed time and wheel speed

The original timestamps and counts remain unchanged. Consecutive count and time differences provide the derived speed estimate.


In [ ]:
data = data.sort_values(["trial_id", "sample_num"]).copy()
data["elapsed_from_step_s"] = (
    data["timestamp_ms"] - data["step_timestamp_ms"]
) / 1000
data["interval_s"] = data.groupby("trial_id")["timestamp_ms"].diff() / 1000
data["count_change"] = data.groupby("trial_id")["encoder_count"].diff()
data["speed_cps"] = data["count_change"] / data["interval_s"]

data[["trial_id", "elapsed_from_step_s", "PWM", "encoder_count", "speed_cps"]].head(10)


## 5. Plot every measured step response

Inspect individual trials before fitting or averaging. The vertical line marks the requested command change.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

sns.lineplot(
    data=data,
    x="elapsed_from_step_s",
    y="PWM",
    hue="trial_id",
    drawstyle="steps-post",
    estimator=None,
    ax=axes[0],
)
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set(title="Requested PWM steps", ylabel="PWM")

sns.lineplot(
    data=data,
    x="elapsed_from_step_s",
    y="speed_cps",
    hue="trial_id",
    estimator=None,
    ax=axes[1],
    legend=False,
)
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set(
    title="Wheel speed calculated from encoder snapshots",
    xlabel="Time from PWM step (s)",
    ylabel="Encoder speed (counts/s)",
)
plt.tight_layout()
plt.show()


## 6. Choose candidate values using fitting trials

These are transient-response models rather than straight-line fits. Their
adjustable coefficients are the slew-rate limit $S$ and the lag time
constant $\tau$:

$$\hat{\omega}_{\mathrm{slew}}(t)=\omega_0+
\operatorname{sign}(\omega_\infty-\omega_0)
\min\left(St,\left|\omega_\infty-\omega_0\right|\right)$$

$$\hat{\omega}_{\mathrm{lag}}(t)=\omega_0+
(\omega_\infty-\omega_0)\left(1-e^{-t/\tau}\right)$$

The notebook estimates $\omega_0$ and $\omega_\infty$ from the initial
and final measured hold periods for each trial. Start with the supplied
values, inspect the next plot, and adjust $S$ and $\tau$ using only trials
labelled `fit`. Once chosen, leave them fixed before inspecting the
held-back trial.


In [ ]:
FIT_TRIAL = "rise_1"
HELD_BACK_TRIAL = "fall_test"
SLEW_RATE_CPS2 = 1500.0
TIME_CONSTANT_S = 0.18

adjustable_coefficients = pd.DataFrame([
    {
        "coefficient": "slew-rate limit",
        "symbol": "S",
        "value": SLEW_RATE_CPS2,
        "units": "counts/s^2",
        "meaning": "largest candidate rate of speed change",
    },
    {
        "coefficient": "time constant",
        "symbol": "tau",
        "value": TIME_CONSTANT_S,
        "units": "s",
        "meaning": "controls how quickly the lag candidate responds",
    },
])

adjustable_coefficients


## 7. Compare candidates with one fitting trial

The initial and final speeds come from the recorded hold periods. The candidate curves describe the transition between those measured levels.


In [ ]:
def add_candidate_predictions(trial, slew_rate, time_constant):
    trial = trial.dropna(subset=["speed_cps"]).copy()
    initial_speed = trial.loc[
        trial["phase"] == "initial_hold", "speed_cps"
    ].median()
    final_speed = trial.loc[
        trial["phase"] == "final_hold", "speed_cps"
    ].median()
    elapsed = np.maximum(trial["elapsed_from_step_s"], 0)
    direction = np.sign(final_speed - initial_speed)

    trial["instantaneous"] = np.where(
        trial["elapsed_from_step_s"] < 0,
        initial_speed,
        final_speed,
    )
    trial["slew_rate"] = initial_speed + direction * np.minimum(
        slew_rate * elapsed,
        abs(final_speed - initial_speed),
    )
    trial["first_order_lag"] = initial_speed + (final_speed - initial_speed) * (
        1 - np.exp(-elapsed / time_constant)
    )
    return trial


fitting_trial = add_candidate_predictions(
    data.loc[data["trial_id"] == FIT_TRIAL],
    SLEW_RATE_CPS2,
    TIME_CONSTANT_S,
)


### Read the coefficients for the selected fitting trial

The two hold-period speeds complete the numerical equations for this trial. They are derived from the measurements rather than adjusted by hand.


In [ ]:
omega_0 = fitting_trial.loc[
    fitting_trial["phase"] == "initial_hold", "speed_cps"
].median()
omega_infinity = fitting_trial.loc[
    fitting_trial["phase"] == "final_hold", "speed_cps"
].median()
direction = np.sign(omega_infinity - omega_0)

trial_coefficients = pd.DataFrame([
    {
        "coefficient": "initial speed",
        "symbol": "omega_0",
        "value": omega_0,
        "units": "counts/s",
        "meaning": "median speed during the initial hold",
    },
    {
        "coefficient": "final speed",
        "symbol": "omega_infinity",
        "value": omega_infinity,
        "units": "counts/s",
        "meaning": "median speed during the final hold",
    },
])

print(
    "Slew candidate: "
    f"omega_hat(t) = {omega_0:.3f} {direction:+.0f} * "
    f"min({SLEW_RATE_CPS2:.3f} * t, {abs(omega_infinity - omega_0):.3f})"
)
print(
    "Lag candidate: "
    f"omega_hat(t) = {omega_0:.3f} "
    f"{omega_infinity - omega_0:+.3f} * "
    f"(1 - exp(-t / {TIME_CONSTANT_S:.3f}))"
)
print(trial_coefficients.to_string(index=False))


### Plot the candidates with the measured response

The equations above and the plotted curves use exactly the same coefficient values.


In [ ]:
fitting_plot = fitting_trial.melt(
    id_vars=["elapsed_from_step_s", "speed_cps"],
    value_vars=["instantaneous", "slew_rate", "first_order_lag"],
    var_name="candidate",
    value_name="predicted_speed_cps",
)

sns.scatterplot(
    data=fitting_trial,
    x="elapsed_from_step_s",
    y="speed_cps",
    color="black",
    s=28,
    label="measured speed",
)
sns.lineplot(
    data=fitting_plot,
    x="elapsed_from_step_s",
    y="predicted_speed_cps",
    hue="candidate",
)
plt.axvline(0, color="black", linestyle="--")
plt.title(f"Candidate responses for fitting trial {FIT_TRIAL}")
plt.xlabel("Time from PWM step (s)")
plt.ylabel("Encoder speed (counts/s)")
plt.show()


## 8. Apply the unchanged candidates to the held-back trial

Do not retune the two candidate values after viewing this result. A falling or differently sized step is a stronger test than another sample from the fitting trace.


In [ ]:
held_back_trial = add_candidate_predictions(
    data.loc[data["trial_id"] == HELD_BACK_TRIAL],
    SLEW_RATE_CPS2,
    TIME_CONSTANT_S,
)

held_back_plot = held_back_trial.melt(
    id_vars=["elapsed_from_step_s", "speed_cps"],
    value_vars=["instantaneous", "slew_rate", "first_order_lag"],
    var_name="candidate",
    value_name="predicted_speed_cps",
)

sns.scatterplot(
    data=held_back_trial,
    x="elapsed_from_step_s",
    y="speed_cps",
    color="black",
    s=28,
    label="measured speed",
)
sns.lineplot(
    data=held_back_plot,
    x="elapsed_from_step_s",
    y="predicted_speed_cps",
    hue="candidate",
)
plt.axvline(0, color="black", linestyle="--")
plt.title(f"Frozen candidates on held-back trial {HELD_BACK_TRIAL}")
plt.xlabel("Time from PWM step (s)")
plt.ylabel("Encoder speed (counts/s)")
plt.show()


## 9. Compare errors without retuning

Mean absolute error gives one compact comparison in the original speed units. Read it alongside the time-series plots: one number can hide whether a candidate misses the beginning or end of the transition.


In [ ]:
error_rows = []
for dataset_name, trial in [
    ("fitting trial", fitting_trial),
    ("held-back trial", held_back_trial),
]:
    for candidate in ["instantaneous", "slew_rate", "first_order_lag"]:
        error_rows.append({
            "data": dataset_name,
            "candidate": candidate,
            "mean_absolute_error_cps": np.mean(
                np.abs(trial["speed_cps"] - trial[candidate])
            ),
        })

pd.DataFrame(error_rows)


## What to notice

- How many speed observations occur while the response is changing?
- Does the trace look instantaneous, slope-limited, curved, or unresolved?
- Which simple candidate misses a repeatable feature of the fitting trial?
- Do the unchanged candidate values remain useful on the held-back transition?
